# Axon exact-v4 multi-tick GPU resume

Attach `axon_exact_v4_bundle.zip` as a Kaggle dataset, enable GPU, then run the cells in order.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

work = Path('/kaggle/working/axon_exact_v4')
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True, exist_ok=True)

bundle_candidates = (
    sorted(Path('/kaggle/input').glob('**/*.axonbundle'))
    + sorted(Path('/kaggle/input').glob('**/*_bundle.zip'))
)
if bundle_candidates:
    bundle = bundle_candidates[0]
    print('bundle zip:', bundle)
    with zipfile.ZipFile(bundle) as zf:
        zf.extractall(work)
else:
    local = Path('/kaggle/working/axon_exact_v4_bundle.axonbundle')
    if not local.exists():
        local = Path('/kaggle/working/axon_exact_v4_bundle.zip')
    if not local.exists():
        local = Path('/kaggle/working/axon_exact_v4_128D_bundle.zip')
    if local.exists():
        print('local bundle zip:', local)
        with zipfile.ZipFile(local) as zf:
            zf.extractall(work)
    else:
        manifest_candidates = sorted(Path('/kaggle/input').glob('**/bundle_manifest.json'))
        if not manifest_candidates:
            raise FileNotFoundError('Attach axon_exact_v4_bundle as a Kaggle dataset first.')
        source_root = manifest_candidates[0].parent
        print('expanded bundle dir:', source_root)
        shutil.copytree(source_root, work, dirs_exist_ok=True)

manifest = json.loads((work / 'bundle_manifest.json').read_text())
print(json.dumps(manifest['checkpoint'], indent=2))
print('dataset:', manifest['dataset'])
print('training:', manifest['training'])

In [ ]:
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi not found')

repo = work / 'axon'
checkpoint_meta = manifest['checkpoint']
training = manifest['training']
dataset_meta = manifest['dataset']
checkpoint = work / checkpoint_meta['archive_path']
dataset_rel = Path(dataset_meta['archive_path'])
if dataset_rel.is_absolute() or '..' in dataset_rel.parts:
    raise RuntimeError(f'unsafe dataset archive path: {dataset_rel}')
dataset_dir = work / dataset_rel
assert repo.exists(), repo
assert checkpoint.exists(), checkpoint
assert dataset_dir.exists(), dataset_dir

start_step = int(training['start_step'])
target_step = int(training['target_step'])
if int(checkpoint_meta['start_step']) != start_step:
    raise RuntimeError('checkpoint and training start_step disagree')
if target_step <= start_step:
    raise RuntimeError(f'invalid leg contract: {start_step} -> {target_step}')
if checkpoint.stat().st_size != int(checkpoint_meta['bytes']):
    raise RuntimeError('source checkpoint byte-size mismatch')

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

verified_file_sha256 = {}
for entry in manifest.get('files', []):
    relative = Path(entry['path'])
    if relative.is_absolute() or '..' in relative.parts:
        raise RuntimeError(f'unsafe bundle file path: {relative}')
    bundled_file = work / relative
    if not bundled_file.is_file() or bundled_file.stat().st_size != int(entry['bytes']):
        raise RuntimeError(f'bundle file size mismatch: {relative}')
    actual_sha256 = sha256_file(bundled_file)
    if actual_sha256 != entry['sha256']:
        raise RuntimeError(f'bundle file SHA-256 mismatch: {relative}')
    verified_file_sha256[relative.as_posix()] = actual_sha256
bundle_manifest_sha256 = sha256_file(work / 'bundle_manifest.json')
checkpoint_sha256 = verified_file_sha256.get(checkpoint_meta['archive_path'])
if checkpoint_sha256 is None:
    checkpoint_sha256 = sha256_file(checkpoint)
if checkpoint_sha256 != checkpoint_meta['sha256']:
    raise RuntimeError('source checkpoint SHA-256 mismatch')

manifest_path = dataset_dir / 'manifest.json'
curriculum_manifest = json.loads(manifest_path.read_text())
curriculum_manifest_sha256 = sha256_file(manifest_path)
if curriculum_manifest_sha256 != dataset_meta['manifest_sha256']:
    raise RuntimeError('exact-v4 curriculum manifest SHA-256 mismatch')
if curriculum_manifest.get('schema') != dataset_meta.get('manifest_schema'):
    raise RuntimeError('exact-v4 curriculum schema mismatch')

# Validate checkpoint continuity where possible.  Legacy phase0b core-only
# checkpoints do not carry optimizer/training state, so the strict contract
# validator may report missing legacy fields.  We log those but continue,
# because the exact-v4 run script starts a fresh optimizer and RNG from the
# bundle manifest.
sys.path.insert(0, str(repo))
import torch
from training.checkpoint_contract import validate_checkpoint_continuity
source_payload = torch.load(checkpoint, map_location='cpu', weights_only=False)
try:
    source_contract = validate_checkpoint_continuity(
        source_payload, expected_step=start_step, expected_families=[], require_cuda_rng=False
    )
    print('checkpoint continuity:', json.dumps(source_contract, indent=2))
except Exception as exc:
    print(f'checkpoint continuity validation skipped: {exc}')
if int(source_payload.get('step', -1)) != start_step:
    raise RuntimeError('checkpoint payload step mismatch')
if 'core_state' not in source_payload or 'cfg' not in source_payload:
    raise RuntimeError('checkpoint missing core_state or cfg')
source_core_config = source_payload['cfg']
del source_payload

print('repo:', repo)
print('checkpoint:', checkpoint)
print('checkpoint sha256:', checkpoint_sha256)
print(f'verified bundle files: {len(verified_file_sha256)} manifest_sha256={bundle_manifest_sha256}')
print('dataset_dir:', dataset_dir)
print('curriculum manifest sha256:', curriculum_manifest_sha256)
print(f'verified leg: {start_step} -> {target_step}')

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is required for this training leg')

run_name = str(training['run_name'])
run_dir = Path('/kaggle/working/runs') / run_name
learning_rate = float(training['lr'])
eval_every = int(training['eval_every'])
checkpoint_every = int(training['checkpoint_every'])
seed = int(training['seed'])
batch_size = int(training.get('batch_size', 1))

# Cross-platform read_view_hash drift workaround.
os.environ['AXON_RELAX_READ_VIEW_HASH'] = '1'

cmd = [
    sys.executable, 'training/run_multitick.py',
    '--checkpoint', str(checkpoint),
    '--dataset-dir', str(dataset_dir),
    '--run-dir', str(run_dir),
    '--device', 'cuda',
    '--steps', str(target_step),
    '--lr', str(learning_rate),
    '--batch-size', str(batch_size),
    '--eval-every', str(eval_every),
    '--eval-episodes', '64',
    '--checkpoint-every', str(checkpoint_every),
    '--seed', str(seed),
]
if bool(training.get('train_soul', False)):
    cmd.append('--train-soul')
    cmd.extend([
        '--soul-continuity-weight',
        str(float(training.get('soul_continuity_weight', 0.01))),
        '--soul-l2-weight',
        str(float(training.get('soul_l2_weight', 1e-5))),
    ])
print(' '.join(cmd))
subprocess.run(cmd, cwd=repo, check=True)

In [ ]:
run_dir = Path('/kaggle/working/runs') / training['run_name']
pointer_path = run_dir / 'pointer.json'
done_path = run_dir / 'checkpoint_done.json'
if not pointer_path.exists() or not done_path.exists():
    raise RuntimeError('trainer did not persist checkpoint sentinels')
pointer = json.loads(pointer_path.read_text())
done = json.loads(done_path.read_text())
final_step = int(pointer['step'])
if int(done['step']) != final_step:
    raise RuntimeError(f'checkpoint sentinels disagree: pointer={final_step} done={done.get("step")}')
completed = final_step == target_step
if not completed:
    print(f'QUARANTINE: expected persisted step {target_step}, got {final_step}')
active_checkpoint = run_dir / pointer['active']
if not active_checkpoint.is_file() or active_checkpoint.stat().st_size == 0:
    raise RuntimeError(f'missing active checkpoint: {active_checkpoint}')
active_sha256 = sha256_file(active_checkpoint)
final_payload = torch.load(active_checkpoint, map_location='cpu', weights_only=False)
if int(final_payload.get('step', -1)) != final_step:
    raise RuntimeError(f'final checkpoint payload step mismatch: {final_payload.get("step")}')
if final_payload.get('cfg') != source_core_config:
    raise RuntimeError('core configuration changed across leg')

leg_result = {
    'schema': 'axon_exact_v4_leg_result_v1',
    'run_name': run_name,
    'start_step': start_step,
    'target_step': target_step,
    'final_step': final_step,
    'completed': completed,
    'source_checkpoint_sha256': checkpoint_sha256,
    'active_checkpoint': pointer['active'],
    'active_checkpoint_bytes': active_checkpoint.stat().st_size,
    'active_checkpoint_sha256': active_sha256,
    'bundle_manifest_sha256': bundle_manifest_sha256,
    'curriculum_manifest_sha256': curriculum_manifest_sha256,
}
leg_path = run_dir / 'leg_result.json'
leg_path.write_text(json.dumps(leg_result, indent=2) + '\n')
Path('/kaggle/working/axon_exact_v4_leg_result.json').write_text(json.dumps(leg_result, indent=2) + '\n')
print(json.dumps(leg_result, indent=2))
print('latest run files:')
for path in sorted(run_dir.glob('*')):
    print(path, path.stat().st_size)